# Atlas integrated scoring

This notebook combines fish use, habitat limitations, biological vulnerability, and population priorities into **relative prioritization indices** for each BSR. It calculates Levels 1 and 2; it does not evaluate individual projects or predict changes in fish abundance.

| Level | Question | What the score represents |
|---|---|---|
| **1. Integrated risk** | Where do fish use and biological vulnerability overlap with impaired habitat conditions? | The sum of condition, fish-use, and vulnerability contributions, weighted by life-stage population priorities. |
| **2. Action alignment** | Which action types are most strongly associated with the limiting factors contributing to that risk? | Existing limiting-factor risk weighted by each action's relationship to the factor. |

The basic unit is one **BSR × species × life stage × limiting factor** pathway. For example, Chinook migration and low summer flows form one pathway within a BSR. The notebook calculates that pathway's contribution, repeats the calculation for all pathways, and sums the results in different ways to support different questions.

**How to read the results:** a larger risk score indicates more overlap among the scored inputs. A larger action-alignment score indicates stronger correspondence between an action and the factors contributing to risk. Neither score is a probability, a physical habitat quantity, or a predicted restoration benefit.

**Run order:** load inputs → validate inputs → transform scores → calculate Levels 1 and 2 → validate calculations → write and verify temporary files → replace the output folder. No published output is replaced until all numerical and file checks pass.

**Input review is separate from numerical QC.** Provisional BSR matches and an unverified condition-rating rubric remain visible even when every mathematical check passes. They are not silently treated as verified.

## 1. Inputs and review settings

Put the five CSVs and polygon GeoPackage in `data/inputs`:

- `Fish Use Scores.csv`
- `Limiting factor scores.csv`
- `Vulnerability table.csv`
- `Population scores.csv`
- `LFAT.csv`
- `bsr.gpkg`

Set `INPUT_DIR_OVERRIDE` only if automatic discovery finds the wrong folder.

The remaining settings record input review, rather than change the scoring equations:

- `BSR_MATCH_REVIEW_NOTES`: optional evidence for independently verified BSR matches, keyed by BSR. Leave empty until correspondence has actually been checked. The source crosswalk status is always preserved.
- `CONDITION_RATING_REVIEW_NOTE`: optional citation or note confirming that **1 means the least impairment and 5 means the greatest impairment** in the source rubric. Leaving this blank identifies the direction as an assumption requiring review.
- `REQUIRE_REVIEWED_INPUTS`: leave `False` for exploratory results with visible review flags. Set `True` when unresolved BSR correspondence or condition-direction review must prevent export. This does not resolve other source uncertainty flags.

The supplied fish-use table marks CC1–CC9 as `provisional_positional`. This notebook cannot verify their correspondence from matching identifiers alone. Do not add a confirmation note solely to clear a flag.

In [ ]:
from contextlib import closing
from pathlib import Path
from uuid import uuid4
import hashlib
import shutil
import sqlite3
import json
import warnings
from datetime import datetime, timezone
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


# Optional: replace None with a folder path if automatic discovery is not appropriate.
INPUT_DIR_OVERRIDE = None

# Enter an evidence-based note only after checking the source BSR correspondence.
# Example structure: {"CC1": "Verified against [source, date, reviewer]."}
BSR_MATCH_REVIEW_NOTES = {}

# Confirm the rating direction against the actual source rubric, not this equation.
CONDITION_RATING_REVIEW_NOTE = ""

# Exploratory exports retain unresolved reviews as explicit fields and metadata.
REQUIRE_REVIEWED_INPUTS = False

FRAMEWORK_VERSION = "2026-09-14.1"
RUN_CREATED_UTC = datetime.now(timezone.utc).isoformat()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
qc_records = []
calculation_qc_passed = False
worked_example_qc_passed = False


def check(condition, name, detail="", stage="input"):
    """Record a passed check or stop immediately with an actionable error."""
    if not bool(condition):
        raise ValueError(f"{stage} QC failed: {name}. {detail}".strip())
    qc_records.append({"stage": stage, "check": name, "result": "passed"})


def compare_frames(actual, expected, name, stage="calculation"):
    """Compare values and columns, allowing only normal serialization rounding."""
    try:
        pd.testing.assert_frame_equal(
            actual.reset_index(drop=True), expected.reset_index(drop=True),
            check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12,
        )
    except AssertionError as error:
        raise ValueError(f"{stage} QC failed: {name}. {error}") from error
    check(True, name, stage=stage)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

INPUT_STEMS = {
    "fish_use": "Fish Use Scores",
    "lfat": "LFAT",
    "limiting_factor": "Limiting factor scores",
    "population": "Population scores",
    "vulnerability": "Vulnerability table",
}
FISH_USE_COLUMNS = [
    "bsr", "basin", "bsr_crosswalk_status",
    "species", "life_stage", "LS_corrected_score",
    "species_aggregate_score", "fish_use_score_decimal",
]
BSR_INPUT_FILE = "bsr.gpkg"
BSR_OUTPUT_FILE = "bsr_scores.gpkg"


def select_input_file(folder, stem, suffix=".csv"):
    # Return one exact or parenthetically numbered input, or None.
    exact = folder / f"{stem}{suffix}"
    if exact.exists():
        return exact
    matches = sorted(folder.glob(f"{stem}(*){suffix}"))
    return matches[0] if len(matches) == 1 else None


def candidate_input_directories(start):
    seen = set()
    for folder in (start, *start.parents):
        for candidate in (
            folder / "data" / "inputs",
            folder / "upload",
            folder,
        ):
            resolved = candidate.resolve()
            if resolved not in seen:
                seen.add(resolved)
                yield resolved


def locate_inputs(start, override=None):
    candidates = [Path(override).expanduser().resolve()] if override else list(
        candidate_input_directories(start)
    )
    for folder in candidates:
        if not folder.is_dir():
            continue
        selected = {
            key: select_input_file(folder, stem)
            for key, stem in INPUT_STEMS.items()
        }
        spatial_path = select_input_file(folder, "bsr", ".gpkg")
        if (
            all(path is not None for path in selected.values())
            and spatial_path is not None
        ):
            return folder, selected, spatial_path
    expected = [
        *(f"{stem}.csv" for stem in INPUT_STEMS.values()),
        BSR_INPUT_FILE,
    ]
    raise FileNotFoundError(
        "Could not find one complete input set. Expected: "
        + ", ".join(expected)
    )


INPUT_DIR, INPUT_PATHS, BSR_INPUT_PATH = locate_inputs(
    Path.cwd().resolve(), INPUT_DIR_OVERRIDE
)
if INPUT_DIR.name == "inputs" and INPUT_DIR.parent.name == "data":
    REPO_ROOT = INPUT_DIR.parent.parent
else:
    REPO_ROOT = Path.cwd().resolve()

OUTPUT_DIR = REPO_ROOT / "data" / "outputs"
QC_DIR = OUTPUT_DIR / "QC"
BSR_OUTPUT_PATH = OUTPUT_DIR / BSR_OUTPUT_FILE

raw = {
    key: pd.read_csv(
        path, usecols=FISH_USE_COLUMNS if key == "fish_use" else None
    )
    for key, path in INPUT_PATHS.items()
}

input_file_hashes = {
    key: file_sha256(path) for key, path in INPUT_PATHS.items()
}
input_file_hashes["spatial"] = file_sha256(BSR_INPUT_PATH)

input_summary = pd.DataFrame(
    [
        {
            "dataset": key,
            "file": path.name,
            "rows": len(raw[key]),
            "columns": len(raw[key].columns),
            "sha256": input_file_hashes[key],
        }
        for key, path in INPUT_PATHS.items()
    ]
)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"QC directory: {QC_DIR}")
print(f"Spatial input: {BSR_INPUT_PATH.name}")
display(input_summary)

## 2. Validate source data before scoring

These checks establish that the input tables can be joined without missing records, accidental duplication, or inconsistent identifiers. Each BSR must contain all 10 species/life-stage combinations and all 15 limiting factors; each action must have a row for every limiting factor, including zero-weight relationships.

The notebook reads only the designated fish-use fields. In particular, overall fish use comes from `fish_use_score_decimal`. Other fish-use score columns in that input are ignored. Required columns may be reordered; unrelated columns in the other CSVs are allowed.

Population priorities are checked within **basin × species**. They distribute weight among a species' life stages; they do not assign explicit relative priorities between species. Source priorities are used as supplied. A total of 0.99 is accepted as rounding and is reported rather than automatically renormalized.

In [ ]:
EXPECTED_COLUMNS = {
    "fish_use": FISH_USE_COLUMNS,
    "lfat": [
        "action_id", "action_type", "action_definition",
        "source_action_label", "limiting_factor",
        "limiting_factor_occurrence", "directness_code",
        "directness_rating", "directness_value", "frequency_code",
        "frequency_rating", "frequency_value", "lfat_score",
        "source_sheet", "source_row", "source_directness_cell",
        "source_frequency_cell", "source_score_cell", "source_notes_cell",
    ],
    "limiting_factor": [
        "bsr", "limiting_factor", "n", "mean_r", "median_r",
        "geo_mean_r", "sd_r", "var_r", "iqr_r", "min_r", "max_r",
        "range_r", "flag_spread", "flag_low_n", "any_flag",
        "lf_condition_score_raw_1_5", "lf_condition_score",
        "condition_transformation",
    ],
    "population": [
        "basin", "species", "life_stage", "population_priority",
    ],
    "vulnerability": [
        "species", "source_life_stage", "life_stage", "limiting_factor",
        "vulnerability_rank", "vulnerability_score", "uncertainty_flag",
        "review_flag", "review_reason", "source_sheet",
        "source_rank_cell", "source_rating_cell", "source_notes_cell",
        "source_uncertainty_cell",
    ],
}

# This fixed list lets later checks confirm complete 15-factor coverage.
CANONICAL_LF = [
    "Anthropogenic Barriers",
    "Riparian Condition",
    "Floodplain Condition",
    "Side Channel and Wetland Habitat",
    "Channel and Habitat Structure",
    "Decreased Water Quantity",
    "Altered Flow Timing",
    "Decreased Sediment Quantity",
    "Increased Sediment Quantity",
    "Summer Water Temperature",
    "Winter Water Temperature",
    "Water Quality",
    "Predation",
    "Altered Primary Productivity",
    "Non-Native Species Interactions and Competition",
]


# Check column names without requiring a particular CSV column order.
schema_rows = []
for name, table in raw.items():
    missing = sorted(set(EXPECTED_COLUMNS[name]) - set(table.columns))
    check(not missing, f"{name}: required columns", f"Missing: {missing}")
    schema_rows.append({"dataset": name, "required_columns": len(EXPECTED_COLUMNS[name]), "schema_pass": True})
schema_qc = pd.DataFrame(schema_rows)

key_columns = {
    "fish_use": ["bsr", "basin", "species", "life_stage", "bsr_crosswalk_status"],
    "population": ["basin", "species", "life_stage"],
    "limiting_factor": ["bsr", "limiting_factor"],
    "vulnerability": ["species", "source_life_stage", "life_stage", "limiting_factor"],
    "lfat": ["action_id", "action_type", "action_definition", "limiting_factor"],
}
for name, columns in key_columns.items():
    for column in columns:
        values = raw[name][column]
        check(values.notna().all(), f"{name}.{column}: no missing identifiers")
        if column != "action_id":
            raw[name][column] = values.astype(str).str.strip()
            check(raw[name][column].ne("").all(), f"{name}.{column}: no blank identifiers")

# Convert only fields used numerically, and reject blanks, text, and infinities.
numeric_columns = {
    "fish_use": ["LS_corrected_score", "species_aggregate_score", "fish_use_score_decimal"],
    "population": ["population_priority"],
    "limiting_factor": ["lf_condition_score_raw_1_5", "lf_condition_score", "n"],
    "vulnerability": ["vulnerability_rank", "vulnerability_score"],
    "lfat": ["directness_value", "frequency_value", "lfat_score"],
}
for name, columns in numeric_columns.items():
    for column in columns:
        values = pd.to_numeric(raw[name][column], errors="coerce")
        check(np.isfinite(values).all(), f"{name}.{column}: finite numeric values")
        raw[name][column] = values

unique_keys = {
    "fish_use": ["bsr", "species", "life_stage"],
    "population": ["basin", "species", "life_stage"],
    "limiting_factor": ["bsr", "limiting_factor"],
    "vulnerability": ["species", "source_life_stage", "limiting_factor"],
    "lfat": ["action_id", "limiting_factor"],
}
for name, keys in unique_keys.items():
    duplicates = raw[name].loc[raw[name].duplicated(keys, keep=False), keys]
    check(duplicates.empty, f"{name}: unique scoring keys", duplicates.head(10).to_dict("records"))

fish = raw["fish_use"]
check(fish["fish_use_score_decimal"].between(0, 1).all(), "Overall fish use is 0–1")
check(fish["LS_corrected_score"].ge(0).all(), "Source life-stage fish use is nonnegative")
check(fish["LS_corrected_score"].max() > 0, "At least one positive life-stage fish-use score")
check(fish["species_aggregate_score"].ge(0).all(), "Species fish use is nonnegative")
for column in ["basin", "fish_use_score_decimal", "bsr_crosswalk_status"]:
    check(fish.groupby("bsr")[column].nunique().eq(1).all(), f"One {column} value per BSR")
check(fish.groupby(["bsr", "species"])["species_aggregate_score"].nunique().eq(1).all(), "One species fish-use value per BSR/species")

for name in ["limiting_factor", "vulnerability", "lfat"]:
    check(set(raw[name]["limiting_factor"]) == set(CANONICAL_LF), f"{name}: canonical limiting factors")
for name, keys in [("limiting_factor", ["bsr"]), ("vulnerability", ["species", "source_life_stage"]), ("lfat", ["action_id"])]:
    check(raw[name].groupby(keys)["limiting_factor"].nunique().eq(15).all(), f"{name}: complete 15-factor coverage")
check(set(fish["bsr"]) == set(raw["limiting_factor"]["bsr"]), "Fish-use and condition BSR coverage agrees")

stage_keys = set(map(tuple, fish[["species", "life_stage"]].drop_duplicates().to_numpy()))
for bsr, group in fish.groupby("bsr"):
    check(set(map(tuple, group[["species", "life_stage"]].to_numpy())) == stage_keys, f"{bsr}: complete species/life-stage coverage")
population_keys = ["basin", "species", "life_stage"]
check(
    set(map(tuple, fish[population_keys].drop_duplicates().to_numpy()))
    == set(map(tuple, raw["population"][population_keys].to_numpy())),
    "Fish-use and population-priority keys agree",
)
check(
    stage_keys == set(map(tuple, raw["vulnerability"][["species", "life_stage"]].drop_duplicates().to_numpy())),
    "Fish-use and vulnerability life stages agree",
)

check(raw["population"]["population_priority"].between(0, 1).all(), "Population priorities are 0–1")
population_review = raw["population"].groupby(["basin", "species"], as_index=False).agg(
    priority_sum=("population_priority", "sum"), life_stage_count=("life_stage", "size")
)
population_review["difference_from_one"] = population_review["priority_sum"] - 1.0
check(np.allclose(population_review["priority_sum"], 1.0, rtol=0, atol=0.011), "Population-priority sums allow only source rounding", population_review.to_dict("records"))
check(raw["limiting_factor"]["lf_condition_score_raw_1_5"].between(1, 5).all(), "Raw condition ratings are 1–5")
check(raw["vulnerability"]["vulnerability_rank"].between(1, 15).all(), "Vulnerability ranks are 1–15")
check(raw["vulnerability"]["vulnerability_rank"].mod(1).eq(0).all(), "Vulnerability ranks are integers")
for column in ["directness_value", "frequency_value", "lfat_score"]:
    check(raw["lfat"][column].between(0, 1).all(), f"LFAT {column} is 0–1")
check(np.allclose(raw["lfat"]["lfat_score"], raw["lfat"]["directness_value"] * raw["lfat"]["frequency_value"], rtol=1e-9, atol=1e-12), "LFAT weight equals directness × frequency")
for column in ["action_type", "action_definition"]:
    check(raw["lfat"].groupby("action_id")[column].nunique().eq(1).all(), f"One {column} per action ID")
check(raw["lfat"].groupby("action_type")["action_id"].nunique().eq(1).all(), "Action types identify unique action IDs")

# Verify expected migration mapping rather than silently collapsing other stages.
for (species, life_stage), group in raw["vulnerability"].groupby(["species", "life_stage"]):
    stages = set(group["source_life_stage"])
    if life_stage == "Migration":
        check(stages == {"Adult Migration & Holding", "Juvenile Emigration"}, f"{species}: adult/juvenile migration source coverage")
    else:
        check(len(stages) == 1, f"{species}/{life_stage}: one uncombined source stage")

check(isinstance(BSR_MATCH_REVIEW_NOTES, dict), "BSR review notes are a dictionary")
check(set(BSR_MATCH_REVIEW_NOTES).issubset(set(fish["bsr"])), "BSR review notes reference existing BSRs")
check(all(isinstance(value, str) and value.strip() for value in BSR_MATCH_REVIEW_NOTES.values()), "BSR review notes contain evidence text")
check(isinstance(CONDITION_RATING_REVIEW_NOTE, str), "Condition-direction review note is text")
check(isinstance(REQUIRE_REVIEWED_INPUTS, bool), "Review requirement is True or False")

display(schema_qc)
display(population_review)

## 3. Put the inputs on their scoring scales

Four inputs enter Level 1. A fifth, the action weight, is used in Level 2.

| Symbol | Input | Used value | Meaning of a larger value |
|---|---|---|---|
| \(F\) | Life-stage fish use | Source `LS_corrected_score` divided by the maximum across the complete input table | More fish use on the source index |
| \(C\) | Limiting-factor condition | \(0.01 + (\text{raw rating}-1)\times0.99/4\) | Greater impairment |
| \(V\) | Biological vulnerability | \(1-(\text{rank}-1)\times0.99/14\) | Greater vulnerability to that limiting factor |
| \(P\) | Population priority | Source priority, unchanged | More weight for this life stage within its basin and species |
| \(W\) | Action relationship weight | Directness × frequency | A stronger action/limiting-factor relationship |


**The 0.01 floor:** the least impaired condition and the lowest vulnerability retain a small nonzero contribution. Zero life-stage fish use still produces zero impact and risk. Linear rank conversion and the nonzero floor are modeling choices, not quantities established by the QC checks.

### Three fish-use measures, with different roles

| Output field | Treatment | Role |
|---|---|---|
| `LS_corrected_score_source` | Original life-stage value | Audit trail |
| `LS_corrected_score` | Source divided by the recorded maximum; 0–1 | The fish-use multiplier in impact and risk |
| `species_aggregate_score` | Source value retained; may exceed 1 | Context only |
| `fish_use_score` | Read exclusively from `fish_use_score_decimal`; 0–1 | Overall BSR context only |

Population priorities remain as supplied, including accepted rounding differences. No separate between-species priority multiplier is introduced.

In [ ]:
# Preserve the source life-stage score and rename BSR fish use for stable
# output naming. BSR fish use is retained for context only.
fish_use = raw["fish_use"].rename(
    columns={
        "LS_corrected_score": "LS_corrected_score_source",
        "fish_use_score_decimal": "fish_use_score",
    }
).copy()

# Normalize life-stage fish use by the maximum value across the complete
# input table. Dividing by the maximum preserves meaningful zeros and the
# ratios among source scores while constraining the multiplier to 0 to 1.
life_stage_fish_use_source_max = fish_use["LS_corrected_score_source"].max()
if (
    not np.isfinite(life_stage_fish_use_source_max)
    or life_stage_fish_use_source_max <= 0
):
    raise ValueError(
        "Life-stage fish-use scores must include at least one positive value."
    )
fish_use["LS_corrected_score"] = (
    fish_use["LS_corrected_score_source"]
    / life_stage_fish_use_source_max
)

# Copy population priorities so the imported table is not modified.
population = raw["population"].copy()

# Preserve the CSV's condition score and give the raw rating a concise name.
condition = raw["limiting_factor"].rename(
    columns={
        "lf_condition_score_raw_1_5": "condition_score_raw_1_5",
        "lf_condition_score": "condition_score_source",
    }
).copy()

# Convert condition linearly: source rating 1 becomes 0.01 and 5 becomes 1.0.
condition["condition_score"] = (
    0.01
    + (condition["condition_score_raw_1_5"] - 1.0) * (0.99 / 4.0)
)

# Preserve the CSV's vulnerability score before recalculating it from rank.
vulnerability = raw["vulnerability"].rename(
    columns={"vulnerability_score": "vulnerability_score_source"}
).copy()

# Convert vulnerability linearly: rank 1 becomes 1.0 and rank 15 becomes 0.01.
vulnerability["vulnerability_score"] = (
    1.0
    - (vulnerability["vulnerability_rank"] - 1.0) * (0.99 / 14.0)
)

# Copy the action crosswalk so derived calculations do not modify the import.
lfat = raw["lfat"].copy()


fish_use["life_stage_fish_use_normalization_max"] = life_stage_fish_use_source_max

# Keep identification evidence separate from numerical validation.
identifier_review = fish_use[["bsr", "basin", "bsr_crosswalk_status"]].drop_duplicates().copy()
identifier_review["bsr_match_review_note"] = identifier_review["bsr"].map(BSR_MATCH_REVIEW_NOTES).fillna("")
identifier_review["bsr_match_review_required"] = (
    identifier_review["bsr_crosswalk_status"].ne("exact_identifier")
    & identifier_review["bsr_match_review_note"].eq("")
)
identifier_review["condition_direction_review_required"] = not bool(CONDITION_RATING_REVIEW_NOTE.strip())
identifier_review["input_review_status"] = np.where(
    identifier_review["bsr_match_review_required"] | identifier_review["condition_direction_review_required"],
    "Provisional: input review required", "BSR matching and condition direction reviewed",
)
identifier_review = identifier_review.sort_values(["basin", "bsr"]).reset_index(drop=True)
fish_use = fish_use.merge(
    identifier_review.drop(columns=["basin", "bsr_crosswalk_status"]),
    on="bsr", how="left", validate="many_to_one",
)

normalization_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "method": "global maximum across complete fish-use input table",
    "source_field": "LS_corrected_score",
    "source_output_field": "LS_corrected_score_source",
    "normalized_output_field": "LS_corrected_score",
    "denominator": float(life_stage_fish_use_source_max),
    "source_min": float(fish_use["LS_corrected_score_source"].min()),
    "normalized_min": float(fish_use["LS_corrected_score"].min()),
    "normalized_max": float(fish_use["LS_corrected_score"].max()),
    "source_rows": len(fish_use),
}])
maximum_rows = fish_use.loc[
    fish_use["LS_corrected_score_source"].eq(life_stage_fish_use_source_max),
    ["bsr", "species", "life_stage", "LS_corrected_score_source"],
]

check(fish_use["LS_corrected_score"].between(0, 1).all(), "Normalized life-stage fish use is 0–1", stage="transformation")
check(np.isclose(fish_use["LS_corrected_score"].max(), 1), "Maximum normalized fish use equals 1", stage="transformation")
check(condition["condition_score"].between(0.01, 1).all(), "Transformed condition is 0.01–1", stage="transformation")
check(vulnerability["vulnerability_score"].between(0.01, 1).all(), "Transformed vulnerability is 0.01–1", stage="transformation")
check(np.allclose(0.01 + (np.array([1., 5.]) - 1) * 0.99 / 4, [0.01, 1.0]), "Condition endpoint anchors", stage="transformation")
check(np.allclose(1 - (np.array([1., 15.]) - 1) * 0.99 / 14, [1.0, 0.01]), "Vulnerability endpoint anchors", stage="transformation")

assumptions = pd.DataFrame([
    ["Life-stage fish use", "LS_corrected_score", "Source divided by the maximum across the complete input table; source and denominator retained."],
    ["Species fish use", "species_aggregate_score", "Unchanged context field; may exceed 1; not a direct multiplier."],
    ["Overall fish use", "fish_use_score_decimal", "Input mapped to output fish_use_score; context only."],
    ["Population priority", "population_priority", "Within-species life-stage weights; source rounding retained; no between-species weighting."],
    ["Condition direction", "condition_score", "Assumes raw 1 means least impairment and 5 greatest impairment; verify against source rubric."],
    ["Condition review evidence", "CONDITION_RATING_REVIEW_NOTE", CONDITION_RATING_REVIEW_NOTE.strip() or "Not supplied; direction remains an assumption requiring review."],
    ["Vulnerability", "vulnerability_score", "Rank 1 maps to 1 and rank 15 to 0.01 using a linear transformation."],
    ["Migration", "maximum vulnerability_score", "Use the larger adult/juvenile vulnerability separately for each species and limiting factor."],
    ["Action weight", "lfat_score", "Directness × frequency; source product checked before scoring."],
    ["Action alignment", "overall_action_alignment_score", "Sum across actions; can count a limiting-factor contribution multiple times; not predicted benefit."],
], columns=["component", "field_or_rule", "implementation"])

print(f"Life-stage fish-use normalization denominator: {life_stage_fish_use_source_max:g}")
display(maximum_rows)
display(identifier_review)
unresolved_bsrs = identifier_review.loc[identifier_review["bsr_match_review_required"], "bsr"].tolist()
if unresolved_bsrs:
    print("BSR correspondence requires review: " + ", ".join(unresolved_bsrs))
if not CONDITION_RATING_REVIEW_NOTE.strip():
    print("Condition direction remains unverified against the source rating rubric.")
if REQUIRE_REVIEWED_INPUTS:
    check(not unresolved_bsrs, "BSR correspondence has been reviewed", unresolved_bsrs, stage="review requirement")
    check(bool(CONDITION_RATING_REVIEW_NOTE.strip()), "Condition direction has been reviewed", stage="review requirement")

## 4. Align migration vulnerability

The vulnerability table contains separate `Adult Migration & Holding` and `Juvenile Emigration` rows, fish use and population priorities contain one `Migration` life stage for Chinook and Steelhead. Vulnerability contains separate adult and juvenile migration records. For each species and limiting factor, use the **larger of the two vulnerability scores**:


$$
\begin{aligned}
\text{combined migration vulnerability}
={}&
\max\Bigl(
\text{adult migration vulnerability},\\
&\qquad \text{juvenile migration vulnerability}
\Bigr)
\end{aligned}
$$


The calculation is applied separately for each species and limiting factor using the recalculated 0.01-to-1.0 vulnerability score. Taking the maximum represents the more vulnerable migration pathway without counting migration twice. Other life stages pass through unchanged.

In [ ]:
def any_yes(values):
    # Replace missing values with blanks and convert all values to text.
    cleaned_values = values.fillna("").astype(str)

    # Ignore surrounding spaces and capitalization when checking for "yes".
    is_yes = cleaned_values.str.strip().str.lower().eq("yes")

    # Return one flag for the full group.
    return "Yes" if is_yes.any() else "No"


# Group by the keys used in scoring. Adult and juvenile migration share the
# combined "Migration" life_stage value and therefore enter the same group.
vulnerability_collapsed = (
    vulnerability.groupby(
        ["species", "life_stage", "limiting_factor"], as_index=False
    )
    .agg(
        # Retain source rank and recalculated-score ranges for review.
        vulnerability_rank_min=("vulnerability_rank", "min"),
        vulnerability_rank_max=("vulnerability_rank", "max"),
        vulnerability_score_min=("vulnerability_score", "min"),
        vulnerability_score_max=("vulnerability_score", "max"),

        # Use the larger adult/juvenile score in the scoring equations.
        vulnerability_score=("vulnerability_score", "max"),

        # Document how many and which source stages contributed.
        source_vulnerability_rows=("source_life_stage", "size"),
        source_life_stages=(
            "source_life_stage",
            lambda values: " | ".join(sorted(set(values.astype(str)))),
        ),

        # Combine review flags and uncertainty notes for the group.
        vulnerability_review_flag=("review_flag", any_yes),
        uncertainty_notes=(
            "uncertainty_flag",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

display(vulnerability_collapsed.head(10))

## 5. Calculate each Level 1 pathway

Let $b$ identify a BSR, $s$ a species, $l$ a life stage, and $f$ a limiting factor. The two pathway scores differ only by population priority:

$$
\text{impact component}
=
\text{life-stage fish use}
\times
\text{limiting-factor condition}
\times
\text{vulnerability}
$$

$$
\text{risk component}
=
\text{impact component}
\times
\text{population priority}
$$

For a given BSR/species/life stage, fish use and population priority stay the same across limiting factors. For a given BSR/limiting factor, condition stays the same across species and life stages. Vulnerability varies with species, life stage, and limiting factor, but not with BSR.

Impact and risk can be aggregated in two equivalent ways:

1. Across the 15 limiting factors for each species and life stage.
2. Across all species and life stages for each limiting factor.

Both approaches produce the same Overall Impact Score and Overall Risk Score within a BSR, although the intermediate scores describe different components of the overall score.

The joins below first attach the population priority to each fish-use row, then expand that row across the relevant vulnerability relationships, and finally attach BSR-specific condition. Missing values or unintended duplicate pathways cause QC to fail before export.

In [ ]:
# Add the applicable population priority to each fish-use row. A left join
# preserves every fish-use row; later QC stops if any priority was not found.
fish_population = fish_use.merge(
    population[["basin", "species", "life_stage", "population_priority"]],
    on=["basin", "species", "life_stage"],
    how="left",
    validate="many_to_one",
)

# Expand each fish-use row across all limiting factors by joining the matching
# vulnerability rows for its species and life stage.
calculation_grid = fish_population.merge(
    vulnerability_collapsed,
    on=["species", "life_stage"],
    how="left",
    validate="many_to_many",
)

# Add the one condition record for the same BSR and limiting factor.
calculation_grid = calculation_grid.merge(
    condition[
        [
            "bsr", "limiting_factor", "condition_score_raw_1_5",
            "condition_score_source", "condition_score", "n", "any_flag",
        ]
    ],
    on=["bsr", "limiting_factor"],
    how="left",
    validate="many_to_one",
)

# Calculate life-stage fish use × vulnerability. This is summed in the
# limiting-factor impact equation.
calculation_grid["fish_vulnerability_component"] = (
    calculation_grid["LS_corrected_score"]
    * calculation_grid["vulnerability_score"]
)

# Calculate condition × vulnerability. This is summed in the life-stage impact
# equation.
calculation_grid["condition_vulnerability_component"] = (
    calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"]
)

# Calculate life-stage fish use × vulnerability × population priority. This is
# summed in the limiting-factor risk equation.
calculation_grid["population_weighted_fish_vulnerability_component"] = (
    calculation_grid["fish_vulnerability_component"]
    * calculation_grid["population_priority"]
)

# Pathway impact = life-stage fish use × condition × vulnerability.
calculation_grid["impact_component"] = (
    calculation_grid["fish_vulnerability_component"]
    * calculation_grid["condition_score"]
)

# Pathway risk = pathway impact × population priority.
calculation_grid["risk_component"] = (
    calculation_grid["impact_component"]
    * calculation_grid["population_priority"]
)

# Display fields needed to audit the first ten pathway calculations.
display(
    calculation_grid[
        [
            "bsr", "species", "life_stage", "limiting_factor",
            "LS_corrected_score_source", "LS_corrected_score",
            "species_aggregate_score",
            "fish_use_score", "population_priority",
            "condition_score_raw_1_5", "condition_score_source",
            "condition_score", "vulnerability_rank_min",
            "vulnerability_rank_max", "vulnerability_score",
            "impact_component", "risk_component",
        ]
    ].head(10)
)

## 6. Summarize Level 1 for different questions

Every summary is a different grouping of the same pathway contributions.

| Summary | Sum across | Question it supports |
|---|---|---|
| Species/life-stage risk | All 15 limiting factors | Which species and life-stage combination contributes most to this BSR's risk? |
| Species risk | That species' life stages and limiting factors | How much risk is associated with each species? |
| Limiting-factor risk | All species and life stages | Which limiting factor contributes most to this BSR's risk? |
| Overall BSR risk | All species, life stages, and limiting factors | What is the combined risk index for this BSR? |

### Life-stage risk scores

For a given BSR, species, and life stage, the Life-Stage Fish Use Score is constant across the 15 limiting factors and can be placed in front of the sum:

$$
\begin{aligned}
\text{life-stage impact score}
={}&
\text{life-stage fish use}
\times
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\Bigl(
\text{limiting-factor condition}
\times
\text{vulnerability}
\Bigr)
\end{aligned}
$$

The population priority is also constant for the applicable basin, species, and life stage:

$$
\begin{aligned}
\text{life-stage risk score}
={}&
\text{life-stage impact score}
\times
\text{population priority}
\end{aligned}
$$

The risk score for the highest-priority life stage is:

$$
\begin{aligned}
\text{risk score for highest priority life stage}
={}&
\max_{\substack{\text{all species and}\\\text{life stages}}}
\left(
\text{life-stage risk score}
\right)
\end{aligned}
$$

The **Highest Priority Life Stage** is the species and life-stage combination with the largest Life-Stage Risk Score within the BSR.

### Species risk scores

For each BSR and species, the species risk scores is the sum of that species' life-stage scores:

$$
\text{species risk score}
=
\sum_{\substack{\text{all life stages}\\\text{for the species}}}
\left(
\text{life-stage risk score}
\right)
$$

The species-level `species_aggregate_score` is retained for context but is not used as a multiplier. Species risk is ranked from largest to smallest within each BSR.

### Limiting-factor risk scores

For a given BSR and limiting factor, the Limiting-Factor Condition is constant across species and life stages and can be placed in front of the sum:

$$
\begin{aligned}
\text{limiting-factor risk score}
={}&
\text{limiting-factor condition}
\times
\sum_{\substack{\text{all species and}\\\text{life stages}}}
\Bigl(
\text{life-stage fish use}
\times
\text{vulnerability}
\times
\text{population priority}
\Bigr)
\end{aligned}
$$

Within this sum, life-stage fish use, vulnerability, and population priority are specific to the applicable species and life stage. Limiting-factor condition is the single condition score for the BSR and limiting factor.

The risk score for the highest-priority limiting factor is:

$$
\begin{aligned}
\text{risk score for highest priority limiting factor}
={}&
\max_{\text{15 limiting factors}}
\left(
\text{limiting-factor risk score}
\right)
\end{aligned}
$$

The **Highest Priority Limiting Factor** is the limiting factor with the largest Limiting-Factor Risk Score within the BSR.

### Overall BSR risk scores

The Overall Risk Score can likewise be calculated equivalently by summing all Life-Stage Risk Scores, all Species Risk Scores, or all 15 Limiting-Factor Risk Scores:

$$
\begin{aligned}
\text{Overall Risk Score}
&=
\sum_{\substack{\text{all species and}\\\text{life stages}}}
\left(
\text{life-stage risk score}
\right) \\
&=
\sum_{\text{all species}}
\left(
\text{species risk score}
\right) \\
&=
\sum_{\text{15 limiting factors}}
\left(
\text{limiting-factor risk score}
\right)
\end{aligned}
$$


In [ ]:
# Group pathway rows by BSR, species, and life stage.
life_stage_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "species", "life_stage"], as_index=False
    )
    .agg(
        # These values are constant within each group, so retain the first.
        LS_corrected_score_source=("LS_corrected_score_source", "first"),
        LS_corrected_score=("LS_corrected_score", "first"),
        life_stage_fish_use_normalization_max=("life_stage_fish_use_normalization_max", "first"),
        species_aggregate_score=("species_aggregate_score", "first"),
        fish_use_score=("fish_use_score", "first"),
        population_priority=("population_priority", "first"),

        # Sum condition × vulnerability across all 15 limiting factors.
        condition_vulnerability_sum=(
            "condition_vulnerability_component", "sum"
        ),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
    )
)

# Apply life-stage fish use to the summed condition/vulnerability contribution.
life_stage_scores["impact_score"] = (
    life_stage_scores["LS_corrected_score"]
    * life_stage_scores["condition_vulnerability_sum"]
)

# Apply population priority to the life-stage impact score.
life_stage_scores["risk_score"] = (
    life_stage_scores["impact_score"]
    * life_stage_scores["population_priority"]
)

# Rank life-stage risk from largest to smallest within each BSR.
life_stage_scores["risk_rank_within_bsr"] = (
    life_stage_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Combine species and life stage into one readable label.
life_stage_scores["species_life_stage_label"] = (
    life_stage_scores["species"] + " | " + life_stage_scores["life_stage"]
)

# Sum life-stage scores for each species within each BSR. The source
# species fish-use score is retained for context and is not a multiplier.
species_scores = (
    life_stage_scores.groupby(
        ["bsr", "basin", "species"], as_index=False
    )
    .agg(
        species_aggregate_score=("species_aggregate_score", "first"),
        fish_use_score=("fish_use_score", "first"),
        life_stage_count=("life_stage", "size"),
        impact_score=("impact_score", "sum"),
        risk_score=("risk_score", "sum"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
    )
)

# Rank species risk from largest to smallest within each BSR.
species_scores["risk_rank_within_bsr"] = (
    species_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Group pathway rows by BSR and limiting factor.
limiting_factor_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "limiting_factor"], as_index=False
    )
    .agg(
        # Retain contextual fields that are constant within each group.
        fish_use_score=("fish_use_score", "first"),
        condition_score_raw_1_5=("condition_score_raw_1_5", "first"),
        condition_score_source=("condition_score_source", "first"),
        condition_score=("condition_score", "first"),
        condition_rating_n=("n", "first"),
        condition_review_flag=("any_flag", "first"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),

        # Sum fish use × vulnerability across species and life stages.
        fish_vulnerability_sum=("fish_vulnerability_component", "sum"),

        # Sum fish use × vulnerability × population priority.
        population_weighted_fish_vulnerability_sum=(
            "population_weighted_fish_vulnerability_component", "sum"
        ),
    )
)

# Apply condition to the summed fish-use/vulnerability contribution.
limiting_factor_scores["impact_score"] = (
    limiting_factor_scores["condition_score"]
    * limiting_factor_scores["fish_vulnerability_sum"]
)

# Apply condition to the population-weighted contribution.
limiting_factor_scores["risk_score"] = (
    limiting_factor_scores["condition_score"]
    * limiting_factor_scores[
        "population_weighted_fish_vulnerability_sum"
    ]
)

# Rank limiting-factor risk from largest to smallest within each BSR.
limiting_factor_scores["risk_rank_within_bsr"] = (
    limiting_factor_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Reduce repeated BSR fish-use values to one contextual row per BSR.
source_fish_use = (
    fish_use.groupby(["bsr", "basin"], as_index=False)
    .agg(
        fish_use_score=("fish_use_score", "first"),
        fish_use_score_variants=("fish_use_score", "nunique"),
    )
)

# Sum all species and life-stage scores to obtain BSR totals.
bsr_from_life_stage = (
    life_stage_scores.groupby(["bsr", "basin"], as_index=False)
    .agg(
        overall_impact_score=("impact_score", "sum"),
        overall_risk_score=("risk_score", "sum"),
    )
)

# Independently sum species scores for the species-level balance check.
bsr_from_species = (
    species_scores.groupby("bsr", as_index=False)
    .agg(
        species_sum_impact_score=("impact_score", "sum"),
        species_sum_risk_score=("risk_score", "sum"),
    )
)

# Independently sum all limiting-factor scores for the balance check.
bsr_from_limiting_factor = (
    limiting_factor_scores.groupby("bsr", as_index=False)
    .agg(
        lf_sum_impact_score=("impact_score", "sum"),
        lf_sum_risk_score=("risk_score", "sum"),
    )
)

# Keep all rank-1 life stages and summarize any ties.
top_life_stage = (
    life_stage_scores.loc[life_stage_scores["risk_rank_within_bsr"].eq(1)]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_species_life_stage=(
            "species_life_stage_label",
            lambda values: "; ".join(sorted(values)),
        ),
        top_species_life_stage_risk_score=("risk_score", "first"),
        top_species_life_stage_risk_tie_count=(
            "species_life_stage_label", "size"
        ),
    )
)

# Keep all rank-1 limiting factors and summarize any ties.
top_limiting_factor = (
    limiting_factor_scores.loc[
        limiting_factor_scores["risk_rank_within_bsr"].eq(1)
    ]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_limiting_factor=(
            "limiting_factor", lambda values: "; ".join(sorted(values))
        ),
        top_limiting_factor_risk_score=("risk_score", "first"),
        top_limiting_factor_risk_tie_count=("limiting_factor", "size"),
    )
)

# Join totals, contextual fish use, and highest-risk contribution summaries.
bsr_scores = (
    bsr_from_life_stage
    .merge(bsr_from_species, on="bsr", validate="one_to_one")
    .merge(bsr_from_limiting_factor, on="bsr", validate="one_to_one")
    .merge(source_fish_use, on=["bsr", "basin"], validate="one_to_one")
    .merge(top_life_stage, on="bsr", validate="one_to_one")
    .merge(top_limiting_factor, on="bsr", validate="one_to_one")
)

# All differences should be zero except for floating-point rounding.
bsr_scores["species_impact_balance_difference"] = (
    bsr_scores["overall_impact_score"]
    - bsr_scores["species_sum_impact_score"]
)
bsr_scores["species_risk_balance_difference"] = (
    bsr_scores["overall_risk_score"]
    - bsr_scores["species_sum_risk_score"]
)
bsr_scores["impact_balance_difference"] = (
    bsr_scores["overall_impact_score"]
    - bsr_scores["lf_sum_impact_score"]
)
bsr_scores["risk_balance_difference"] = (
    bsr_scores["overall_risk_score"]
    - bsr_scores["lf_sum_risk_score"]
)

display(
    bsr_scores[
        [
            "bsr", "overall_risk_score",
            "highest_risk_species_life_stage",
            "highest_risk_limiting_factor",
            "fish_use_score",
        ]
    ].sort_values("overall_risk_score", ascending=False).head(10)
)

bsr_scores = bsr_scores.merge(identifier_review, on=["bsr", "basin"], how="left", validate="one_to_one")
bsr_scores["life_stage_fish_use_normalization_max"] = life_stage_fish_use_source_max
bsr_scores["scoring_run_id"] = RUN_ID
bsr_scores["scoring_framework_version"] = FRAMEWORK_VERSION

## 7. Calculate Level 2 action alignment

The action weight is calculated from relationship directness and frequency:

$$
\text{action weight}
=
\text{relationship directness}
\times
\text{frequency}
$$


For each action, the action-specific benefit components are summed across the 15 limiting factors:

$$
\begin{aligned}
\text{action-specific benefit score}
={}&
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\left(
\text{limiting-factor risk score}
\times
\text{action weight}
\right)
\end{aligned}
$$

The Overall Benefit Score is the sum of the Action-Specific Benefit Scores across all actions:

$$
\begin{aligned}
\text{overall benefit score}
={}&
\sum_{\text{all actions}}
\left(
\text{action-specific benefit score}
\right)
\end{aligned}
$$

The **Highest Risk-Aligned Action Type** is the action type with the largest Action-Specific Benefit Score within the BSR.

### Existing app compatibility

The clearer fields are added alongside the fields required by the existing Streamlit app. Each pair contains exactly the same values; no second score is calculated.

| Preferred field | Existing app field retained as an alias |
|---|---|
| `condition_action_alignment_score` | `condition_improvement_score` |
| `impact_action_alignment_score` | `limiting_factor_amelioration_score` |
| `action_alignment_score` | `action_benefit_score` |
| `overall_condition_action_alignment_score` | `overall_condition_improvement_score` |
| `overall_impact_action_alignment_score` | `overall_limiting_factor_amelioration_score` |
| `overall_action_alignment_score` | `overall_benefit_score` |

The notebook exports a field dictionary and a standalone framework explanation under `QC`. An app that hardcodes the old wording must update its display text to use those explanations. Numeric compatibility alone does not update app labels or make it display the new BSR review fields.

In [ ]:
# Join each limiting-factor score to every action related to that factor.
# Each result is one BSR × limiting factor × action row.
action_components = limiting_factor_scores.merge(
    lfat[
        [
            "action_id", "action_type", "action_definition",
            "limiting_factor", "directness_code", "directness_value",
            "frequency_code", "frequency_value", "lfat_score",
        ]
    ],
    on="limiting_factor",
    how="inner",
    validate="many_to_many",
)

# Condition–action alignment component = condition × action weight.
action_components["condition_improvement_component"] = (
    action_components["condition_score"]
    * action_components["lfat_score"]
)

# Impact–action alignment component = impact × action weight.
action_components["amelioration_component"] = (
    action_components["impact_score"]
    * action_components["lfat_score"]
)

# Action alignment component = risk × action weight. Retain the app field name.
action_components["benefit_component"] = (
    action_components["risk_score"]
    * action_components["lfat_score"]
)

# Sum all 15 limiting-factor contributions for each BSR and action.
action_scores = (
    action_components.groupby(
        [
            "bsr", "basin", "action_id", "action_type",
            "action_definition",
        ],
        as_index=False,
    )
    .agg(
        condition_improvement_score=(
            "condition_improvement_component", "sum"
        ),
        limiting_factor_amelioration_score=(
            "amelioration_component", "sum"
        ),
        action_benefit_score=("benefit_component", "sum"),

        # Count condition and vulnerability records flagged for review.
        condition_review_count=(
            "condition_review_flag",
            lambda values: int(
                values.fillna(False)
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["true", "yes", "1"])
                .sum()
            ),
        ),
        vulnerability_review_count=(
            "vulnerability_review_flag",
            lambda values: int(values.eq("Yes").sum()),
        ),
    )
)

# Add an action-specific display label.
action_scores["benefit_score_label"] = (
    action_scores["action_type"].astype(str) + " action alignment score"
)

# Rank action benefit from largest to smallest within each BSR.
action_scores["benefit_rank_within_bsr"] = (
    action_scores.groupby("bsr")["action_benefit_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Keep all rank-1 actions and summarize any ties.
top_action = (
    action_scores.loc[action_scores["benefit_rank_within_bsr"].eq(1)]
    .assign(
        priority_action_label=lambda table: (
            table["action_id"].astype(str) + " | " + table["action_type"]
        )
    )
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_aligned_action_type=(
            "priority_action_label",
            lambda values: "; ".join(sorted(values)),
        ),
        highest_action_benefit_score=("action_benefit_score", "first"),
        top_action_benefit_tie_count=("priority_action_label", "size"),
    )
)

# Sum across actions. These totals measure alignment and can count a
# limiting-factor contribution repeatedly through different action types.
overall_action_scores = (
    action_scores.groupby("bsr", as_index=False)
    .agg(
        overall_condition_improvement_score=(
            "condition_improvement_score", "sum"
        ),
        overall_limiting_factor_amelioration_score=(
            "limiting_factor_amelioration_score", "sum"
        ),
        overall_benefit_score=("action_benefit_score", "sum"),
    )
)

# Add action summaries to the BSR output.
bsr_scores = (
    bsr_scores
    .merge(top_action, on="bsr", validate="one_to_one")
    .merge(overall_action_scores, on="bsr", validate="one_to_one")
)


# Preferred names make the index interpretation explicit. Legacy names remain
# as equal-valued aliases so the existing app can read this output schema.
ACTION_FIELD_ALIASES = {
    "condition_action_alignment_score": "condition_improvement_score",
    "impact_action_alignment_score": "limiting_factor_amelioration_score",
    "action_alignment_score": "action_benefit_score",
}
BSR_FIELD_ALIASES = {
    "overall_condition_action_alignment_score": "overall_condition_improvement_score",
    "overall_impact_action_alignment_score": "overall_limiting_factor_amelioration_score",
    "overall_action_alignment_score": "overall_benefit_score",
    "highest_action_alignment_score": "highest_action_benefit_score",
}
for preferred, existing in ACTION_FIELD_ALIASES.items():
    action_scores[preferred] = action_scores[existing]
for preferred, existing in BSR_FIELD_ALIASES.items():
    bsr_scores[preferred] = bsr_scores[existing]
action_components["action_alignment_component"] = action_components["benefit_component"]

action_factor_weights = lfat.groupby("limiting_factor", as_index=False).agg(
    summed_action_weight=("lfat_score", "sum"),
    positive_weight_actions=("lfat_score", lambda values: int(values.gt(0).sum())),
    listed_actions=("action_id", "nunique"),
).sort_values("summed_action_weight", ascending=False).reset_index(drop=True)

display(action_factor_weights)
display(action_scores[["bsr", "action_type", "condition_action_alignment_score", "impact_action_alignment_score", "action_alignment_score", "benefit_rank_within_bsr"]].sort_values(["bsr", "benefit_rank_within_bsr"]).head(15))
display(bsr_scores[["bsr", "overall_risk_score", "overall_action_alignment_score", "input_review_status"]].sort_values("overall_risk_score", ascending=False).head(10))

## 8. Validate the completed calculations

The checks independently sum pathway components and compare them with the life-stage, species, limiting-factor, and BSR summaries. They also verify every action total, zero-fish-use behavior, output aliases, expected record coverage, and finite scores.

A passing result means the tables and equations agree. It does not establish that provisional BSR matches, condition-rating assumptions, vulnerability ranks, or action weights are biologically correct. Those review items remain separately identified.

In [ ]:
# All checks in this cell run before any output file is staged or published.
calculation_qc_passed = False
worked_example_qc_passed = False
stage = "calculation"
check(len(vulnerability_collapsed) == len(stage_keys) * 15, "Collapsed vulnerability row count", stage=stage)
check(not vulnerability_collapsed.duplicated(["species", "life_stage", "limiting_factor"]).any(), "Unique collapsed vulnerability keys", stage=stage)
check(len(calculation_grid) == len(fish_use) * 15, "Complete pathway-grid row count", stage=stage)
check(not calculation_grid.duplicated(["bsr", "species", "life_stage", "limiting_factor"]).any(), "Unique pathway keys", stage=stage)
check(calculation_grid.groupby("bsr").size().eq(len(stage_keys) * 15).all(), "Complete pathway coverage in each BSR", stage=stage)
for column in ["LS_corrected_score", "species_aggregate_score", "fish_use_score", "population_priority", "condition_score", "vulnerability_score", "impact_component", "risk_component"]:
    check(np.isfinite(calculation_grid[column]).all(), f"Finite joined {column}", stage=stage)
check(np.allclose(calculation_grid["impact_component"], calculation_grid["LS_corrected_score"] * calculation_grid["condition_score"] * calculation_grid["vulnerability_score"], rtol=1e-12, atol=1e-12), "Pathway impact equation", stage=stage)
check(np.allclose(calculation_grid["risk_component"], calculation_grid["impact_component"] * calculation_grid["population_priority"], rtol=1e-12, atol=1e-12), "Pathway risk equation", stage=stage)
zero = calculation_grid["LS_corrected_score"].eq(0)
check(calculation_grid.loc[zero, ["impact_component", "risk_component"]].eq(0).all().all(), "Zero fish use gives zero pathway scores", stage=stage)

# Rebuild each summary directly from pathway components rather than copying
# the factorized equations used in the production calculation cells.
for table_name, table, keys in [
    ("life stage", life_stage_scores, ["bsr", "basin", "species", "life_stage"]),
    ("species", species_scores, ["bsr", "basin", "species"]),
    ("limiting factor", limiting_factor_scores, ["bsr", "basin", "limiting_factor"]),
]:
    expected = calculation_grid.groupby(keys, as_index=False).agg(
        impact_score=("impact_component", "sum"), risk_score=("risk_component", "sum")
    ).sort_values(keys)
    actual = table[keys + ["impact_score", "risk_score"]].sort_values(keys)
    compare_frames(actual, expected, f"{table_name} scores equal pathway sums")

keys = ["bsr", "basin", "species", "life_stage"]
detail_fields = ["LS_corrected_score_source", "LS_corrected_score", "species_aggregate_score", "life_stage_fish_use_normalization_max"]
compare_frames(life_stage_scores[keys + detail_fields].sort_values(keys), fish_use[keys + detail_fields].sort_values(keys), "Source and normalized fish-use fields preserved")
for column in ["species_impact_balance_difference", "species_risk_balance_difference", "impact_balance_difference", "risk_balance_difference"]:
    check(np.allclose(bsr_scores[column], 0, rtol=0, atol=1e-12), f"{column} within rounding tolerance", stage=stage)
check(set(bsr_scores["bsr"]) == set(fish_use["bsr"]), "BSR summary coverage", stage=stage)
check(not bsr_scores["bsr"].duplicated().any(), "One summary row per BSR", stage=stage)
check(len(action_components) == len(bsr_scores) * len(lfat), "Complete action-component coverage", stage=stage)
check(len(action_scores) == len(bsr_scores) * lfat["action_id"].nunique(), "Complete BSR/action coverage", stage=stage)
check(not action_scores.duplicated(["bsr", "action_id"]).any(), "Unique BSR/action keys", stage=stage)

for component, input_field in [("condition_improvement_component", "condition_score"), ("amelioration_component", "impact_score"), ("benefit_component", "risk_score")]:
    check(np.allclose(action_components[component], action_components[input_field] * action_components["lfat_score"], rtol=1e-12, atol=1e-12), f"{component} equation", stage=stage)
action_keys = ["bsr", "action_id"]
expected_actions = action_components.groupby(action_keys, as_index=False).agg(
    condition_improvement_score=("condition_improvement_component", "sum"),
    limiting_factor_amelioration_score=("amelioration_component", "sum"),
    action_benefit_score=("benefit_component", "sum"),
)
action_fields = list(ACTION_FIELD_ALIASES.values())
compare_frames(action_scores[action_keys + action_fields].sort_values(action_keys), expected_actions.sort_values(action_keys), "Action scores equal component sums")
expected_overall = action_scores.groupby("bsr", as_index=False).agg(
    overall_condition_improvement_score=("condition_improvement_score", "sum"),
    overall_limiting_factor_amelioration_score=("limiting_factor_amelioration_score", "sum"),
    overall_benefit_score=("action_benefit_score", "sum"),
)
compare_frames(bsr_scores[expected_overall.columns].sort_values("bsr"), expected_overall.sort_values("bsr"), "Overall alignment equals the sum across actions")

# Independently use the effective factor weights to reproduce the overall total.
effective = limiting_factor_scores.merge(action_factor_weights[["limiting_factor", "summed_action_weight"]], on="limiting_factor", validate="many_to_one")
effective["contribution"] = effective["risk_score"] * effective["summed_action_weight"]
effective_total = effective.groupby("bsr")["contribution"].sum().sort_index()
check(np.allclose(effective_total, bsr_scores.set_index("bsr")["overall_action_alignment_score"].sort_index(), rtol=1e-12, atol=1e-12), "Overall action total equals factor risk × summed action weight", stage=stage)
for table, aliases in [(action_scores, ACTION_FIELD_ALIASES), (bsr_scores, BSR_FIELD_ALIASES)]:
    for preferred, existing in aliases.items():
        check(table[preferred].equals(table[existing]), f"Compatible alias: {preferred}", stage=stage)
for name, table in [("life stage", life_stage_scores), ("species", species_scores), ("limiting factor", limiting_factor_scores), ("action", action_scores), ("BSR", bsr_scores)]:
    scores = [column for column in table if column.endswith("_score")]
    check(np.isfinite(table[scores].to_numpy(dtype=float)).all(), f"{name}: finite output scores", stage=stage)

def scoring_state_signature():
    """Detect table/configuration edits made after validation in a live kernel."""
    digest = hashlib.sha256()
    for name in ["fish_use", "population", "condition", "vulnerability", "lfat", "vulnerability_collapsed", "calculation_grid", "life_stage_scores", "species_scores", "limiting_factor_scores", "action_components", "action_scores", "bsr_scores", "identifier_review"]:
        table = globals()[name]
        digest.update(name.encode("utf-8"))
        digest.update(repr(table.columns.tolist()).encode("utf-8"))
        digest.update(pd.util.hash_pandas_object(table, index=True).to_numpy().tobytes())
    digest.update(json.dumps({"bsr_notes": BSR_MATCH_REVIEW_NOTES, "condition_note": CONDITION_RATING_REVIEW_NOTE, "require_reviewed_inputs": REQUIRE_REVIEWED_INPUTS, "normalization_max": float(life_stage_fish_use_source_max)}, sort_keys=True).encode("utf-8"))
    return digest.hexdigest()


validated_calculation_signature = scoring_state_signature()
calculation_qc_passed = True
print(f"Input, transformation, and calculation checks passed: {len(qc_records)}.")
print(f"Validated {len(bsr_scores)} BSRs, {len(calculation_grid):,} pathways, and {lfat['action_id'].nunique()} action types.")
print("Input review flags remain separate from this numerical result.")

## 9. A small example to check the logic

Consider two life stages, two limiting factors, and two actions. Spawning has zero fish use, so its contributions must all be zero. Rearing has fish use 0.8 and population priority 0.4. For rearing, condition × vulnerability totals $0.10\times0.50+1.00\times1.00=1.05$.

Therefore **rearing impact = 0.8 × 1.05 = 0.84**, and **rearing risk = 0.84 × 0.4 = 0.336**. Grouping those contributions by limiting factor or species must produce the same totals. Applying the two action-weight sets gives action-alignment scores 0.328 and 0.176; their overall alignment total is 0.504. The larger overall action total reflects repeated action relationships, not additional independently realized benefit.

The example below uses explicit expected results and runs before export. It illustrates the equations with simplified values, not an actual BSR.

In [ ]:
worked_example_qc_passed = False

# Define two life stages and their population priorities.
test_population = {"Spawning": 0.60, "Rearing": 0.40}

# This BSR fish-use value is contextual and is not used below.
test_fish_use_score = 0.75

# These life-stage scores represent normalized 0-to-1 fish-use multipliers.
test_life_stage_fish_use = {"Spawning": 0.00, "Rearing": 0.80}

# Define two condition scores and pathway-specific vulnerabilities.
test_condition = {"Temperature": 0.10, "Instream Complexity": 1.00}
test_vulnerability = {
    ("Spawning", "Temperature"): 1.00,
    ("Rearing", "Temperature"): 0.50,
    ("Spawning", "Instream Complexity"): 0.50,
    ("Rearing", "Instream Complexity"): 1.00,
}
test_action_weight = {
    ("Floodplain Restoration", "Temperature"): 0.50,
    ("Floodplain Restoration", "Instream Complexity"): 1.00,
    ("Riparian Planting", "Temperature"): 1.00,
    ("Riparian Planting", "Instream Complexity"): 0.50,
}

# Calculate one impact and risk component for each pathway.
test_rows = []
for life_stage in test_population:
    for limiting_factor, condition_score in test_condition.items():
        # Impact = life-stage fish use × condition × vulnerability.
        impact = (
            test_life_stage_fish_use[life_stage]
            * condition_score
            * test_vulnerability[(life_stage, limiting_factor)]
        )

        # Risk = impact × population priority.
        risk = impact * test_population[life_stage]

        # Store each pathway so it can be aggregated in two directions.
        test_rows.append(
            {
                "life_stage": life_stage,
                "species": "Test Salmon",
                "limiting_factor": limiting_factor,
                "fish_use_score": test_fish_use_score,
                "LS_corrected_score": test_life_stage_fish_use[life_stage],
                "condition_score": condition_score,
                "impact_component": impact,
                "risk_component": risk,
            }
        )
test_grid = pd.DataFrame(test_rows)

# Sum by life stage and independently by limiting factor.
test_life = test_grid.groupby("life_stage", as_index=False).agg(
    impact_score=("impact_component", "sum"),
    risk_score=("risk_component", "sum"),
)
test_species = test_grid.groupby("species", as_index=False).agg(
    impact_score=("impact_component", "sum"),
    risk_score=("risk_component", "sum"),
)
test_lf = test_grid.groupby("limiting_factor", as_index=False).agg(
    condition_score=("condition_score", "first"),
    impact_score=("impact_component", "sum"),
    risk_score=("risk_component", "sum"),
)

# Apply action weights to condition, impact, and risk.
test_actions = []
for action in ["Floodplain Restoration", "Riparian Planting"]:
    rows = test_lf.copy()
    rows["weight"] = rows["limiting_factor"].map(
        lambda factor: test_action_weight[(action, factor)]
    )
    test_actions.append(
        {
            "action": action,
            "benefit_score_label": f"{action} action alignment score",
            "condition_improvement_score": (
                rows["condition_score"] * rows["weight"]
            ).sum(),
            "limiting_factor_amelioration_score": (
                rows["impact_score"] * rows["weight"]
            ).sum(),
            "action_benefit_score": (
                rows["risk_score"] * rows["weight"]
            ).sum(),
        }
    )
test_actions = pd.DataFrame(test_actions)

# Sum the action-specific scores across actions.
test_overall_action_scores = pd.Series(
    {
        "overall_condition_improvement_score": (
            test_actions["condition_improvement_score"].sum()
        ),
        "overall_limiting_factor_amelioration_score": (
            test_actions["limiting_factor_amelioration_score"].sum()
        ),
        "overall_benefit_score": test_actions["action_benefit_score"].sum(),
    }
)

# Check the life-stage results.
expected_life = {
    "Spawning": (0.00, 0.00),
    "Rearing": (0.84, 0.336),
}
for row in test_life.itertuples(index=False):
    expected_impact, expected_risk = expected_life[row.life_stage]
    assert np.isclose(row.impact_score, expected_impact)
    assert np.isclose(row.risk_score, expected_risk)

# Check the contextual field, zero-score rule, and aggregation balance.
assert test_grid["fish_use_score"].eq(test_fish_use_score).all()
zero_rows = test_grid["LS_corrected_score"].eq(0)
assert test_grid.loc[zero_rows, "impact_component"].eq(0).all()
assert test_grid.loc[zero_rows, "risk_component"].eq(0).all()
assert np.isclose(test_life["impact_score"].sum(), 0.84)
assert np.isclose(test_life["risk_score"].sum(), 0.336)
assert np.isclose(test_species["impact_score"].sum(), 0.84)
assert np.isclose(test_species["risk_score"].sum(), 0.336)
assert np.isclose(
    test_life["impact_score"].sum(), test_lf["impact_score"].sum()
)
assert np.isclose(
    test_life["risk_score"].sum(), test_lf["risk_score"].sum()
)

# Check action-specific and overall results.
assert np.allclose(
    test_actions["condition_improvement_score"], [1.05, 0.60]
)
assert np.allclose(
    test_actions["limiting_factor_amelioration_score"], [0.82, 0.44]
)
assert np.allclose(test_actions["action_benefit_score"], [0.328, 0.176])
assert np.isclose(
    test_overall_action_scores["overall_condition_improvement_score"], 1.65
)
assert np.isclose(
    test_overall_action_scores["overall_limiting_factor_amelioration_score"],
    1.26,
)
assert np.isclose(
    test_overall_action_scores["overall_benefit_score"], 0.504
)

worked_example_qc_passed = True
print("Worked example passed. Output-file checks have not yet run.")
check(True, "Hand-calculated example", stage="calculation")
display(test_life)
display(test_species)
display(test_lf)
display(test_actions)
display(test_overall_action_scores.to_frame("score"))

## 10. Prepare export tables and their documentation

The eight existing core CSV filenames and `bsr_scores.gpkg` are retained. BSR outputs now include crosswalk status, review notes, unresolved-review flags, the fish-use normalization denominator, and the run/version identifiers. Clearer action-alignment fields coexist with the existing app fields.

`QC` contains the pathway support tables, BSR and vulnerability review information, normalization details, population-priority sums, effective action weights, field definitions, a framework explanation, input-file hashes, and the QC record for this run. These files document both the implemented calculation and unresolved interpretation questions.

The scored GeoPackage preserves the input geometry and spatial index and includes nonspatial fish-use, population, life-stage, species, limiting-factor, and action tables for joins. The spatial output must contain exactly the scored BSRs.

In [ ]:
CORE_SCORE_FILES = {
    "bsr": "bsr_scores.csv",
    "fish_use": "fish_use_scores.csv",
    "population": "population_scores.csv",
    "life_stage": "life_stage_scores.csv",
    "species": "species_scores.csv",
    "limiting_factor": "limiting_factor_scores_integrated.csv",
    "action": "action_scores.csv",
    "grid": "calculation_grid.csv",
}

CORE_OUTPUTS = {
    CORE_SCORE_FILES["bsr"]: bsr_scores,
    CORE_SCORE_FILES["fish_use"]: fish_use,
    CORE_SCORE_FILES["population"]: population,
    CORE_SCORE_FILES["life_stage"]: life_stage_scores,
    CORE_SCORE_FILES["species"]: species_scores,
    CORE_SCORE_FILES["limiting_factor"]: limiting_factor_scores,
    CORE_SCORE_FILES["action"]: action_scores,
    CORE_SCORE_FILES["grid"]: calculation_grid,
}
QC_OUTPUTS = {
    "action_score_components.csv": action_components,
    "assumptions_for_review.csv": assumptions,
    "bsr_identifiers_for_review.csv": identifier_review,
    "vulnerability_scores_for_review.csv": vulnerability_collapsed,
}

GPKG_ATTRIBUTE_TABLES = {
    "fish_use_scores": {
        "table": fish_use,
        "identifier": "Atlas fish-use scores",
        "description": (
            "Source and normalized overall, species, and life-stage fish-use scores"
        ),
    },
    "population_scores": {
        "table": population,
        "identifier": "Atlas population scores",
        "description": (
            "Source population-priority scores by basin, species, and life stage"
        ),
    },
    "life_stage_scores": {
        "table": life_stage_scores,
        "identifier": "Atlas life-stage scores",
        "description": (
            "Level 1 species and life-stage scores by BSR for attribute joins"
        ),
    },
    "species_scores": {
        "table": species_scores,
        "identifier": "Atlas species scores",
        "description": (
            "Level 1 species impact and risk scores by BSR"
        ),
    },
    "limiting_factor_scores": {
        "table": limiting_factor_scores,
        "identifier": "Atlas limiting-factor scores",
        "description": (
            "Level 1 limiting-factor scores by BSR for attribute joins"
        ),
    },
    "action_type_scores": {
        "table": action_scores,
        "identifier": "Atlas action-type scores",
        "description": (
            "Level 2 action-alignment indices by BSR; existing app aliases retained"
        ),
    },
}


# A concise field dictionary provides recommended display labels and meanings.
field_definitions = pd.DataFrame([
    ["fish_use / life_stage / grid", "LS_corrected_score_source", "Source Life-Stage Fish Use", "Unchanged input value before maximum normalization.", ""],
    ["fish_use / life_stage / grid", "LS_corrected_score", "Life-Stage Fish Use Score", "Source divided by the recorded global maximum; 0–1; risk multiplier.", ""],
    ["fish_use / life_stage / bsr / grid", "life_stage_fish_use_normalization_max", "Life-Stage Fish-Use Normalization Maximum", "Denominator used for this run; compare before interpreting changes between runs.", ""],
    ["fish_use / life_stage / species", "species_aggregate_score", "Species Fish Use Score", "Unchanged context field; may exceed 1; not a multiplier.", ""],
    ["fish_use / bsr", "fish_use_score", "Overall Fish Use Score", "Input fish_use_score_decimal, unchanged; 0–1; context only.", ""],
    ["limiting_factor / grid", "condition_score", "Limiting-Factor Impairment Score", "Raw 1–5 mapped to 0.01–1; assumes higher raw values indicate greater impairment.", ""],
    ["population / life_stage / grid", "population_priority", "Life-Stage Population Priority", "Within basin/species weight; source rounding retained; no explicit between-species multiplier.", ""],
    ["bsr", "overall_risk_score", "Overall Risk Score", "Sum of population-weighted pathway contributions; relative index, not a probability.", ""],
    ["action", "condition_action_alignment_score", "Condition–Action Alignment Score", "Sum of condition × action weight; not a predicted improvement.", "condition_improvement_score"],
    ["action", "impact_action_alignment_score", "Impact–Action Alignment Score", "Sum of limiting-factor impact × action weight.", "limiting_factor_amelioration_score"],
    ["action", "action_alignment_score", "Action Alignment Score", "Sum of limiting-factor risk × action weight for one action.", "action_benefit_score"],
    ["bsr", "overall_condition_action_alignment_score", "Overall Condition–Action Alignment Score", "Sum across actions; includes repeated factor relationships.", "overall_condition_improvement_score"],
    ["bsr", "overall_impact_action_alignment_score", "Overall Impact–Action Alignment Score", "Sum across actions; includes repeated factor relationships.", "overall_limiting_factor_amelioration_score"],
    ["bsr", "overall_action_alignment_score", "Overall Action Alignment Score", "Sum across actions; repeated factor relationships; not additive realized benefits.", "overall_benefit_score"],
    ["bsr", "highest_action_alignment_score", "Highest Action Alignment Score", "Largest action-specific alignment score within a BSR.", "highest_action_benefit_score"],
    ["bsr / fish_use / grid", "bsr_crosswalk_status", "Source BSR Crosswalk Status", "Original source status preserved, including provisional_positional.", ""],
    ["bsr / fish_use / grid", "bsr_match_review_required", "BSR Correspondence Requires Review", "Non-exact source match without an evidence-based review note.", ""],
    ["bsr / fish_use / grid", "condition_direction_review_required", "Condition Direction Requires Review", "True when no source-rubric confirmation note was supplied.", ""],
], columns=["tables", "field", "recommended_label", "definition", "existing_app_alias"])

run_metadata = {
    "run_id": RUN_ID,
    "created_utc": RUN_CREATED_UTC,
    "framework_version": FRAMEWORK_VERSION,
    "life_stage_fish_use_normalization": normalization_summary.iloc[0].to_dict(),
    "normalization_maximum_source_rows": maximum_rows.to_dict("records"),
    "condition_rating_assumption": "1 = least impairment; 5 = greatest impairment",
    "condition_rating_review_note": CONDITION_RATING_REVIEW_NOTE.strip(),
    "condition_direction_review_required": not bool(CONDITION_RATING_REVIEW_NOTE.strip()),
    "unresolved_bsr_matches": unresolved_bsrs,
    "bsr_match_review_notes": BSR_MATCH_REVIEW_NOTES,
    "require_reviewed_inputs": REQUIRE_REVIEWED_INPUTS,
    "population_priorities": "Source values retained; sums checked within basin/species with absolute tolerance 0.011.",
    "run_review_status": "provisional" if identifier_review["input_review_status"].str.startswith("Provisional").any() else "BSR correspondence and condition direction reviewed",
    "source_vulnerability_review_rows": int(vulnerability["review_flag"].astype(str).str.strip().str.casefold().eq("yes").sum()),
    "source_condition_review_rows": int(condition["any_flag"].astype(str).str.strip().str.casefold().isin(["true", "yes", "1"]).sum()),
    "input_files": {key: {"file_name": path.name, "sha256": input_file_hashes[key]} for key, path in {**INPUT_PATHS, "spatial": BSR_INPUT_PATH}.items()},
    "core_output_rows": {name: len(table) for name, table in CORE_OUTPUTS.items()},
    "numerical_qc": "passed before staging",
}
QC_OUTPUTS.update({
    "input_file_summary.csv": input_summary,
    "life_stage_fish_use_normalization.csv": normalization_summary,
    "normalization_maximum_source_rows.csv": maximum_rows,
    "population_priority_sums.csv": population_review,
    "action_factor_weight_totals.csv": action_factor_weights,
    "score_field_definitions.csv": field_definitions,
})

In [ ]:
# Framework text exported to QC/scoring_framework.md.
# This is a copy of the notebook narrative for use in downstream displays.
# If the framework narrative is edited, keep this export text aligned.
FRAMEWORK_EXPLANATION_MARKDOWN = r"""
# Atlas integrated scoring

This notebook combines fish use, habitat limitations, biological vulnerability, and population priorities into **relative prioritization indices** for each BSR. It calculates Levels 1 and 2; it does not evaluate individual projects or predict changes in fish abundance.

| Level | Question | What the score represents |
|---|---|---|
| **1. Integrated risk** | Where do fish use and biological vulnerability overlap with impaired habitat conditions? | The sum of condition, fish-use, and vulnerability contributions, weighted by life-stage population priorities. |
| **2. Action alignment** | Which action types are most strongly associated with the limiting factors contributing to that risk? | Existing limiting-factor risk weighted by each action's relationship to the factor. |

The basic unit is one **BSR × species × life stage × limiting factor** pathway. For example, Chinook migration and low summer flows form one pathway within a BSR. The notebook calculates that pathway's contribution, repeats the calculation for all pathways, and sums the results in different ways to support different questions.

**How to read the results:** a larger risk score indicates more overlap among the scored inputs. A larger action-alignment score indicates stronger correspondence between an action and the factors contributing to risk. Neither score is a probability, a physical habitat quantity, or a predicted restoration benefit.

**Run order:** load inputs → validate inputs → transform scores → calculate Levels 1 and 2 → validate calculations → write and verify temporary files → replace the output folder. No published output is replaced until all numerical and file checks pass.

**Input review is separate from numerical QC.** Provisional BSR matches and an unverified condition-rating rubric remain visible even when every mathematical check passes. They are not silently treated as verified.

## 3. Put the inputs on their scoring scales

Four inputs enter Level 1. A fifth, the action weight, is used in Level 2.

| Symbol | Input | Used value | Meaning of a larger value |
|---|---|---|---|
| \(F\) | Life-stage fish use | Source `LS_corrected_score` divided by the maximum across the complete input table | More fish use on the source index |
| \(C\) | Limiting-factor condition | \(0.01 + (\text{raw rating}-1)\times0.99/4\) | Greater impairment, **assuming the source rubric has this direction** |
| \(V\) | Biological vulnerability | \(1-(\text{rank}-1)\times0.99/14\) | Greater vulnerability to that limiting factor |
| \(P\) | Population priority | Source priority, unchanged | More weight for this life stage within its basin and species |
| \(W\) | Action relationship weight | Directness × frequency | A stronger action/limiting-factor relationship |

**Condition anchors:** the implemented assumption is 1 = least impairment and 5 = greatest impairment. This is a model assumption until checked against the source rubric. If the rubric uses the opposite direction, correct the transformation before using these outputs for decisions. A descriptive field name alone is not sufficient evidence.

**The 0.01 floor:** the least impaired condition and the lowest vulnerability retain a small nonzero contribution. Zero life-stage fish use still produces zero impact and risk. Linear rank conversion and the nonzero floor are modeling choices, not quantities established by the QC checks.

### Three fish-use measures, with different roles

| Output field | Treatment | Role |
|---|---|---|
| `LS_corrected_score_source` | Original life-stage value | Audit trail |
| `LS_corrected_score` | Source divided by the recorded maximum; 0–1 | The fish-use multiplier in impact and risk |
| `species_aggregate_score` | Source value retained; may exceed 1 | Context only |
| `fish_use_score` | Read exclusively from `fish_use_score_decimal`; 0–1 | Overall BSR context only |

The same field name, `LS_corrected_score`, refers to the **source value in the input** and the **normalized value in the outputs**. The source is therefore also retained under an explicit source name. Each relevant output records `life_stage_fish_use_normalization_max` so the transformation can be reconstructed.

**Comparison between runs:** normalization uses one maximum across both basins and all species/life stages. If that maximum changes, impact, risk, and risk-based action scores for otherwise unchanged records all rescale. Ratios and rankings among unchanged records are preserved by this common scaling. Absolute scores from runs with different denominators should not be compared without accounting for the change.

Population priorities remain as supplied, including accepted rounding differences. No separate between-species priority multiplier is introduced.

## 4. Align migration vulnerability

Fish use and population priorities contain one `Migration` life stage for Chinook and Steelhead. Vulnerability contains separate adult and juvenile migration records. For each species and limiting factor, use the **larger of the two vulnerability scores**:

$$V_{\mathrm{migration}}=\max(V_{\mathrm{adult}},V_{\mathrm{juvenile}}).$$

For example, adult vulnerability 0.8 and juvenile vulnerability 0.4 produce a combined score of 0.8. This represents the more vulnerable migration pathway without adding migration twice. It is not an average. Other life stages pass through unchanged. Both source stages, their score range, and their review flags remain in the review table.

## 5. Calculate each Level 1 pathway

Let $b$ identify a BSR, $s$ a species, $l$ a life stage, and $f$ a limiting factor. The two pathway scores differ only by population priority:

$$I_{bslf}=F_{bsl}\,C_{bf}\,V_{slf}$$

$$R_{bslf}=I_{bslf}\,P_{\mathrm{basin}(b),s,l}.$$

**Impact** is the unweighted overlap of life-stage fish use, impairment, and vulnerability. **Risk** adds the population-priority weight. These are names for relative indices; risk is not a probability of loss.

For a given BSR/species/life stage, fish use and population priority stay the same across limiting factors. For a given BSR/limiting factor, condition stays the same across species and life stages. Vulnerability varies with species, life stage, and limiting factor, but not with BSR.

The joins below first attach the population priority to each fish-use row, then expand that row across the relevant vulnerability relationships, and finally attach BSR-specific condition. Missing values or unintended duplicate pathways cause QC to fail before export.

## 6. Summarize Level 1 for different questions

Every summary is a different grouping of the same pathway contributions.

| Summary | Sum across | Question it supports |
|---|---|---|
| Species/life-stage risk | All 15 limiting factors | Which species and life-stage combination contributes most to this BSR's risk? |
| Species risk | That species' life stages and limiting factors | How much risk is associated with each species? |
| Limiting-factor risk | All species and life stages | Which limiting factor contributes most to this BSR's risk? |
| Overall BSR risk | All species, life stages, and limiting factors | What is the combined risk index for this BSR? |

The two principal calculation routes are:

$$R_{bsl}=F_{bsl}\,P_{\mathrm{basin}(b),s,l}\sum_f C_{bf}V_{slf},\qquad
R_{bf}=C_{bf}\sum_{s,l}F_{bsl}V_{slf}P_{\mathrm{basin}(b),s,l}.$$

They must give the same overall total:

$$R_b=\sum_{s,l}R_{bsl}=\sum_s R_{bs}=\sum_f R_{bf}.$$

Impact uses the same groupings without population priority. Aggregate scores are sums and can exceed 1. Species and overall fish-use context fields do not enter these sums as additional multipliers.

The `highest_risk_*` fields identify the largest contributions, not a complete restoration-priority decision. Equal scores share a dense rank, and every tied top label is retained. If every contribution is zero, the labels describe a zero-score tie rather than evidence of a high-priority condition.

## 7. Calculate Level 2 action alignment

For action $a$ and limiting factor $f$, the source action relationship weight is $W_{fa}=\mathrm{directness}\times\mathrm{frequency}$. The input checks verify this product before it is used.

The notebook applies that weight to three increasingly integrated inputs:

| Recommended name | Equation for one BSR/action | What it includes |
|---|---|---|
| **Condition–Action Alignment Score** | $\sum_f C_{bf}W_{fa}$ | Current impairment and action relationships |
| **Impact–Action Alignment Score** | $\sum_f I_{bf}W_{fa}$ | Adds fish use and vulnerability |
| **Action Alignment Score** | $A_{ba}=\sum_f R_{bf}W_{fa}$ | Also includes population priority |

These indices do **not** estimate the amount of condition improvement, habitat gain, or fish response. They do not account for cost, feasibility, implementation constraints, landowner willingness, or site-specific effectiveness. Adding a species or an action can change aggregate totals; keep the evaluated framework consistent when comparing runs.

### What the overall action total means

$$A_b=\sum_a A_{ba}=\sum_f R_{bf}\left(\sum_a W_{fa}\right).$$

This is the **Overall Action Alignment Score**. A limiting factor receives more effective weight if it is associated with more actions or stronger action relationships. Therefore the total can rank BSRs differently from Overall Risk, and it is not an additive estimate of realized benefits. The table displayed below reports each factor's summed action weight so this effect is visible.

### Existing app compatibility

The clearer fields are added alongside the fields required by the existing Streamlit app. Each pair contains exactly the same values; no second score is calculated.

| Preferred field | Existing app field retained as an alias |
|---|---|
| `condition_action_alignment_score` | `condition_improvement_score` |
| `impact_action_alignment_score` | `limiting_factor_amelioration_score` |
| `action_alignment_score` | `action_benefit_score` |
| `overall_condition_action_alignment_score` | `overall_condition_improvement_score` |
| `overall_impact_action_alignment_score` | `overall_limiting_factor_amelioration_score` |
| `overall_action_alignment_score` | `overall_benefit_score` |

The notebook exports a field dictionary and a standalone framework explanation under `QC`. An app that hardcodes the old wording must update its display text to use those explanations. Numeric compatibility alone does not update app labels or make it display the new BSR review fields.
"""

### GeoPackage helper functions

These functions copy the source GeoPackage, add the score and review fields, and register the nonspatial score tables. They do not transform or rewrite polygon geometry. Existing spatial-index triggers are restored after attribute updates, and the next section checks geometry, feature IDs, index rows, and trigger definitions against the source.

The helper writes only to the temporary export location supplied by the final cell. A failure here leaves the published outputs in place.

In [ ]:
def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


def sqlite_type(series):
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    if pd.api.types.is_numeric_dtype(series):
        return "REAL"
    return "TEXT"


def sqlite_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def geometry_digest(path, feature_table, key_field, geometry_field):
    digest = hashlib.sha256()
    query = (
        f"SELECT {quote_identifier(key_field)}, "
        f"{quote_identifier(geometry_field)} "
        f"FROM {quote_identifier(feature_table)} "
        f"ORDER BY {quote_identifier(key_field)}"
    )
    with closing(sqlite3.connect(path)) as connection:
        for key, geometry in connection.execute(query):
            digest.update(str(key).encode("utf-8"))
            digest.update(bytes(geometry) if geometry is not None else b"")
    return digest.hexdigest()


def write_attribute_table(
    connection, table_name, table, identifier, description
):
    columns = table.columns.tolist()
    normalized_columns = [str(column).lower() for column in columns]
    if len(normalized_columns) != len(set(normalized_columns)):
        raise ValueError(
            f"GeoPackage table {table_name} has duplicate column names."
        )
    if "fid" in normalized_columns:
        raise ValueError(
            f"GeoPackage table {table_name} already contains a fid field."
        )
    table_exists = connection.execute(
        "SELECT 1 FROM sqlite_master WHERE name = ?", (table_name,)
    ).fetchone()
    if table_exists is not None:
        raise ValueError(
            f"GeoPackage already contains a table named {table_name}."
        )

    column_definitions = ["fid INTEGER PRIMARY KEY AUTOINCREMENT"]
    column_definitions.extend(
        f"{quote_identifier(column)} {sqlite_type(table[column])}"
        for column in columns
    )
    connection.execute(
        f"CREATE TABLE {quote_identifier(table_name)} "
        f"({', '.join(column_definitions)})"
    )

    placeholders = ", ".join("?" for _ in columns)
    insert_sql = (
        f"INSERT INTO {quote_identifier(table_name)} "
        f"({', '.join(quote_identifier(column) for column in columns)}) "
        f"VALUES ({placeholders})"
    )
    connection.executemany(
        insert_sql,
        (
            tuple(sqlite_value(value) for value in row)
            for row in table.itertuples(index=False, name=None)
        ),
    )
    connection.execute(
        "INSERT INTO gpkg_contents "
        "(table_name, data_type, identifier, description, last_change) "
        "VALUES (?, 'attributes', ?, ?, "
        "strftime('%Y-%m-%dT%H:%M:%fZ', 'now'))",
        (table_name, identifier, description),
    )
    if "bsr" in normalized_columns:
        bsr_column = columns[normalized_columns.index("bsr")]
        index_name = f"idx_{table_name}_bsr"
        connection.execute(
            f"CREATE INDEX {quote_identifier(index_name)} "
            f"ON {quote_identifier(table_name)} "
            f"({quote_identifier(bsr_column)})"
        )


def write_scored_bsr_gpkg(
    input_path, output_path, summary, attribute_tables
):
    # sqlite3 connections are closed explicitly because its context
    # manager commits or rolls back but does not close the file handle.
    # An open handle prevents Path.replace() on Windows.
    with closing(sqlite3.connect(input_path)) as connection:
        feature_tables = [
            row[0]
            for row in connection.execute(
                "SELECT table_name FROM gpkg_contents WHERE data_type = 'features'"
            )
        ]
        candidates = []
        for table_name in feature_tables:
            columns = [
                row[1]
                for row in connection.execute(
                    f"PRAGMA table_info({quote_identifier(table_name)})"
                )
            ]
            source_key = next(
                (column for column in columns if column.lower() == "bsr"),
                None,
            )
            if source_key is not None:
                candidates.append((table_name, source_key, columns))

        if len(candidates) != 1:
            raise ValueError(
                "Expected exactly one feature layer containing a BSR field; "
                f"found {len(candidates)}."
            )
        feature_table, source_key, source_columns = candidates[0]
        geometry_row = connection.execute(
            "SELECT column_name FROM gpkg_geometry_columns WHERE table_name = ?",
            (feature_table,),
        ).fetchone()
        if geometry_row is None:
            raise ValueError(
                f"No geometry field is registered for layer {feature_table}."
            )
        geometry_field = geometry_row[0]
        spatial_keys = {
            str(row[0]).strip()
            for row in connection.execute(
                f"SELECT {quote_identifier(source_key)} "
                f"FROM {quote_identifier(feature_table)}"
            )
        }

    summary_keys = set(summary["bsr"].astype(str).str.strip())
    if spatial_keys != summary_keys:
        raise ValueError(
            "The GeoPackage and score summary have different BSR coverage. "
            f"Missing scores: {sorted(spatial_keys - summary_keys)}; "
            f"missing geometry: {sorted(summary_keys - spatial_keys)}"
        )

    scored = summary.rename(columns={"bsr": "score_bsr"}).copy()
    scored_columns = scored.columns.tolist()
    existing_lower = {column.lower() for column in source_columns}
    collisions = [
        column for column in scored_columns if column.lower() in existing_lower
    ]
    if collisions:
        raise ValueError(
            "Score fields collide with existing GeoPackage fields: "
            + ", ".join(collisions)
        )

    temporary_path = output_path.with_name(
        f".{output_path.stem}.{uuid4().hex}.tmp.gpkg"
    )
    shutil.copy2(input_path, temporary_path)

    try:
        with closing(sqlite3.connect(temporary_path)) as connection:
            # Only attributes are updated. Suspend existing RTree triggers
            # during those updates, then restore their exact SQL. Geometry,
            # feature IDs, index rows, and trigger definitions are checked
            # against the source before this staged file can be published.
            prefix = f"rtree_{feature_table}_{geometry_field}_"
            spatial_triggers = [
                (name, sql) for name, sql in connection.execute(
                    "SELECT name, sql FROM sqlite_master WHERE type='trigger' AND tbl_name=?",
                    (feature_table,),
                ) if name.startswith(prefix)
            ]
            with connection:
                for trigger_name, _ in spatial_triggers:
                    connection.execute(f"DROP TRIGGER {quote_identifier(trigger_name)}")
                for column in scored_columns:
                    connection.execute(
                        f"ALTER TABLE {quote_identifier(feature_table)} "
                        f"ADD COLUMN {quote_identifier(column)} "
                        f"{sqlite_type(scored[column])}"
                    )

                assignments = ", ".join(
                    f"{quote_identifier(column)} = ?"
                    for column in scored_columns
                )
                update_sql = (
                    f"UPDATE {quote_identifier(feature_table)} "
                    f"SET {assignments} "
                    f"WHERE TRIM(CAST("
                    f"{quote_identifier(source_key)} AS TEXT)) = ?"
                )
                for original_bsr, (_, row) in zip(
                    summary["bsr"].astype(str), scored.iterrows()
                ):
                    values = [
                        sqlite_value(row[column])
                        for column in scored_columns
                    ]
                    values.append(original_bsr.strip())
                    cursor = connection.execute(update_sql, values)
                    if cursor.rowcount != 1:
                        raise ValueError(
                            f"Expected one spatial row for {original_bsr}; "
                            f"updated {cursor.rowcount}."
                        )

                connection.execute(
                    "UPDATE gpkg_contents "
                    "SET identifier = ?, description = ?, last_change = strftime('%Y-%m-%dT%H:%M:%fZ', 'now') "
                    "WHERE table_name = ?",
                    (
                        "Atlas scored BSRs",
                        "BSR geometry with Level 1 risk, Level 2 alignment, and input-review status",
                        feature_table,
                    ),
                )

                for table_name, table_specification in attribute_tables.items():
                    write_attribute_table(
                        connection=connection,
                        table_name=table_name,
                        **table_specification,
                    )

                for _, trigger_sql in spatial_triggers:
                    connection.execute(trigger_sql)

            integrity = connection.execute(
                "PRAGMA integrity_check"
            ).fetchone()[0]
            if integrity != "ok":
                raise ValueError(
                    f"GeoPackage integrity check failed: {integrity}"
                )

        # Path.replace() overwrites an existing output atomically. The SQLite
        # connection must be closed before this line on Windows.
        try:
            temporary_path.replace(output_path)
        except PermissionError as error:
            raise PermissionError(
                f"Could not replace {output_path}. Close the output "
                "GeoPackage in QGIS, ArcGIS, or another application, then "
                "run this cell again."
            ) from error
    finally:
        # Do not let cleanup of a staging file hide the original exception.
        try:
            temporary_path.unlink(missing_ok=True)
        except OSError:
            pass

    return (
        feature_table, geometry_field, source_key, scored_columns,
        list(attribute_tables),
    )

## 11. Verify temporary files, then publish the complete output set

This final cell stages the new files beside `data/outputs` and reads them back. It checks every exported table, the scored polygon attributes, registered attribute tables, database integrity, geometry bytes, feature IDs, the spatial index, and unchanged input-file hashes.

Only after these checks pass does the notebook replace the output directory. The prior directory is kept as a temporary backup during replacement and restored if replacement fails. Files unrelated to this notebook are preserved. Run only one export at a time; on Windows, close the output GeoPackage in GIS software if it is locked.

The final message reports **numerical QC** and **input review status** separately. A provisional result can be computationally correct while its source correspondence or rating interpretation still needs review.

In [ ]:
def read_csv_for_comparison(path, expected):
    """Preserve text identifiers and blank audit notes during CSV readback."""
    text_columns = {
        column: str for column in expected.columns
        if pd.api.types.is_object_dtype(expected[column].dtype)
        or pd.api.types.is_string_dtype(expected[column].dtype)
    }
    loaded = pd.read_csv(path, dtype=text_columns)
    for column in text_columns:
        if expected[column].notna().all():
            loaded[column] = loaded[column].fillna("")
    return loaded


def spatial_index_snapshot(path, feature_table, geometry_field):
    prefix = f"rtree_{feature_table}_{geometry_field}"
    with closing(sqlite3.connect(path)) as connection:
        exists = connection.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name=?", (prefix,)).fetchone()
        rows = connection.execute(f"SELECT * FROM {quote_identifier(prefix)} ORDER BY id").fetchall() if exists else None
        triggers = sorted(
            (name, sql) for name, sql in connection.execute(
                "SELECT name, sql FROM sqlite_master WHERE type='trigger' AND tbl_name=?", (feature_table,)
            ) if name.startswith(prefix + "_")
        )
    return rows, triggers


def publish_output_directory(staged_directory, target_directory):
    """Replace a validated directory, restoring the old one on a failed move."""
    backup = target_directory.with_name(f".{target_directory.name}.{RUN_ID}.backup")
    had_previous = target_directory.exists()
    if had_previous:
        target_directory.rename(backup)
    try:
        staged_directory.rename(target_directory)
    except Exception as publish_error:
        if had_previous:
            try:
                backup.rename(target_directory)
            except Exception as restore_error:
                raise RuntimeError(
                    f"Output replacement and automatic restore failed. Previous outputs remain at {backup}. "
                    f"Restore error: {restore_error}"
                ) from publish_error
        raise RuntimeError("Output replacement failed; previous outputs were restored. Close any locked output files and rerun this cell.") from publish_error
    if had_previous:
        try:
            shutil.rmtree(backup)
        except OSError:
            warnings.warn(f"New outputs were published, but the previous backup could not be removed: {backup}")


check(calculation_qc_passed, "Calculation QC completed", stage="staging")
check(worked_example_qc_passed, "Worked example completed", stage="staging")
check(scoring_state_signature() == validated_calculation_signature, "Scoring tables and settings unchanged since validation", "Run the notebook from the beginning after editing inputs or calculations.", stage="staging")
# Recheck source hashes immediately before export to detect mid-run changes.
for name, path in {**INPUT_PATHS, "spatial": BSR_INPUT_PATH}.items():
    check(file_sha256(path) == input_file_hashes[name], f"Input unchanged: {path.name}", stage="staging")

OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(prefix=f".atlas_{RUN_ID}_", dir=OUTPUT_DIR.parent) as temporary_root:
    staged = Path(temporary_root) / "outputs"
    if OUTPUT_DIR.exists():
        # Preserve unrelated files while replacing every output owned by this run.
        shutil.copytree(OUTPUT_DIR, staged)
    else:
        staged.mkdir()
    staged_qc = staged / "QC"
    staged_qc.mkdir(exist_ok=True)

    for name, table in CORE_OUTPUTS.items():
        table.to_csv(staged / name, index=False)
    for name, table in QC_OUTPUTS.items():
        table.to_csv(staged_qc / name, index=False)

    staged_gpkg = staged / BSR_OUTPUT_FILE
    feature_table, geometry_field, source_key, score_fields, attribute_names = write_scored_bsr_gpkg(
        BSR_INPUT_PATH, staged_gpkg, bsr_scores, GPKG_ATTRIBUTE_TABLES
    )

    # Check values in every staged core and supporting CSV, not just its existence.
    for directory, tables in [(staged, CORE_OUTPUTS), (staged_qc, QC_OUTPUTS)]:
        for name, expected in tables.items():
            actual = read_csv_for_comparison(directory / name, expected)
            compare_frames(actual, expected, f"CSV readback: {name}", stage="staged output")

    with closing(sqlite3.connect(staged_gpkg)) as connection:
        check(connection.execute("PRAGMA integrity_check").fetchone()[0] == "ok", "GeoPackage database integrity", stage="staged output")
        check(not connection.execute("PRAGMA foreign_key_check").fetchall(), "GeoPackage foreign-key integrity", stage="staged output")
        query = f"SELECT {', '.join(quote_identifier(column) for column in score_fields)} FROM {quote_identifier(feature_table)} ORDER BY score_bsr"
        actual = pd.read_sql_query(query, connection)
        expected = bsr_scores.rename(columns={"bsr": "score_bsr"})[score_fields].sort_values("score_bsr")
        compare_frames(actual, expected, "GeoPackage BSR scores and review fields", stage="staged output")
        registered = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents WHERE data_type='attributes'")}
        check(set(attribute_names).issubset(registered), "GeoPackage attribute tables registered", stage="staged output")
        for name, specification in GPKG_ATTRIBUTE_TABLES.items():
            expected = specification["table"]
            columns = ", ".join(quote_identifier(column) for column in expected.columns)
            actual = pd.read_sql_query(f"SELECT {columns} FROM {quote_identifier(name)} ORDER BY fid", connection)
            compare_frames(actual, expected, f"GeoPackage attribute values: {name}", stage="staged output")
        null_geometry = connection.execute(f"SELECT COUNT(*) FROM {quote_identifier(feature_table)} WHERE {quote_identifier(geometry_field)} IS NULL").fetchone()[0]
        check(null_geometry == 0, "No null scored geometry", stage="staged output")
        feature_id = next(row[1] for row in connection.execute(f"PRAGMA table_info({quote_identifier(feature_table)})") if row[5] == 1)

    check(geometry_digest(BSR_INPUT_PATH, feature_table, source_key, geometry_field) == geometry_digest(staged_gpkg, feature_table, source_key, geometry_field), "Geometry and BSR identifiers preserved byte-for-byte", stage="staged output")
    check(geometry_digest(BSR_INPUT_PATH, feature_table, feature_id, geometry_field) == geometry_digest(staged_gpkg, feature_table, feature_id, geometry_field), "Geometry and feature IDs preserved byte-for-byte", stage="staged output")
    check(spatial_index_snapshot(BSR_INPUT_PATH, feature_table, geometry_field) == spatial_index_snapshot(staged_gpkg, feature_table, geometry_field), "Spatial-index rows and trigger definitions preserved", stage="staged output")

    # Bind this output set to the exact input versions used in the calculation.
    for name, path in {**INPUT_PATHS, "spatial": BSR_INPUT_PATH}.items():
        check(file_sha256(path) == input_file_hashes[name], f"Input still unchanged: {path.name}", stage="staged output")
    run_metadata["numerical_qc"] = "input, transformation, calculation, example, and staged-file checks passed"
    metadata_path = staged_qc / "scoring_run_metadata.json"
    metadata_path.write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    check(json.loads(metadata_path.read_text(encoding="utf-8")) == run_metadata, "Run metadata readback", stage="staged output")
    methodology_path = staged_qc / "scoring_framework.md"
    methodology_path.write_text(FRAMEWORK_EXPLANATION_MARKDOWN, encoding="utf-8")
    check(methodology_path.read_text(encoding="utf-8") == FRAMEWORK_EXPLANATION_MARKDOWN, "Framework explanation readback", stage="staged output")

    # Write the final QC record after all substantive staging checks are complete.
    qc_summary = pd.DataFrame(qc_records)
    qc_summary.to_csv(staged_qc / "qc_summary.csv", index=False)
    pd.testing.assert_frame_equal(pd.read_csv(staged_qc / "qc_summary.csv"), qc_summary)
    publish_output_directory(staged, OUTPUT_DIR)

print(f"Published validated outputs to: {OUTPUT_DIR}")
print(f"Run: {RUN_ID}; numerical and file QC checks passed: {len(qc_summary)}.")
print(f"Input review status: {run_metadata['run_review_status']}.")
if unresolved_bsrs:
    print("BSR correspondence still requires review: " + ", ".join(unresolved_bsrs))
if not CONDITION_RATING_REVIEW_NOTE.strip():
    print("Condition-rating direction still requires confirmation against the source rubric.")
print("Source uncertainty flags remain documented; passing numerical QC does not resolve them.")
display(pd.DataFrame([{"file": name, "rows": len(table)} for name, table in CORE_OUTPUTS.items()]))